## Part 1 – Build the Classifier

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC

In [16]:
corpus = [
    "the movie was fantastic and i loved every part of it",
    "an absolute masterpiece with brilliant acting",
    "the film was boring and too long",
    "i really enjoyed the story and the visuals",
    "the plot was terrible and the acting was even worse",
    "what a wonderful experience, highly recommend",
    "not worth my time, very disappointing",
    "a truly great film, i will watch it again",
    "the script was weak and the characters were flat",
    "an amazing journey from start to finish"
]

categories = [
    "Positive", "Positive", "Negative", "Positive", "Negative",
    "Positive", "Negative", "Positive", "Negative", "Positive"
]

test_corpus = [
    "the movie was great",
    "i hated the film",
    "a boring and bad story",
    "absolutely loved it"
]

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(corpus)
X_test = vectorizer.transform(test_corpus)

model = SVC()
model.fit(X_train, categories)  


predictions = model.predict(X_test)

print("\n--- Predictions ---")
for sentence, label in zip(test_corpus, predictions):
    print(f"'{sentence}' --> {label}")

correct_labels = ["Positive", "Negative", "Negative", "Positive"]

correct_count = 0
for predicted, actual in zip(predictions, correct_labels):
    if predicted == actual:
        correct_count += 1

print(f"\nCorrectly classified: {correct_count} out of {len(correct_labels)}")


--- Predictions ---
'the movie was great' --> Positive
'i hated the film' --> Positive
'a boring and bad story' --> Positive
'absolutely loved it' --> Positive

Correctly classified: 2 out of 4


In [8]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Apeksha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
vectorizer = CountVectorizer(ngram_range=(1,2))

X_train = vectorizer.fit_transform(corpus)
X_test = vectorizer.transform(test_corpus)

model = SVC()
model.fit(X_train, categories)

predictions = model.predict(X_test)

print("--- Predictions with Bigrams ---")
for sentence, label in zip(test_corpus, predictions):
    print(f"'{sentence}' --> {label}")

correct_count = sum(1 for p, a in zip(predictions, correct_labels) if p == a)
print(f"\nCorrectly classified: {correct_count} out of {len(correct_labels)}")

--- Predictions with Bigrams ---
'the movie was great' --> Positive
'i hated the film' --> Positive
'a boring and bad story' --> Positive
'absolutely loved it' --> Positive

Correctly classified: 2 out of 4


## Part 2 – Investigate the Model

**Q1: What happens if you change ngram_range from (1,1) to (1,2)?**
When we change ngram_range to (1,2) the vectorizer captures both single words 
and pairs of consecutive words. For example "not good" becomes its own feature 
instead of being split into "not" and "good" separately. However in our case 
the accuracy stayed the same at 2 out of 4 because the dataset was too small 
to benefit from the extra features.

**Q2: Why might bigrams help in sentiment analysis?**
Bigrams help because some meanings only make sense as word pairs. For example 
"not good" means something completely different from "good" alone. A unigram 
model splits these apart and loses the meaning, but a bigram model keeps them 
together as one feature. This is especially useful for capturing negations and 
phrases like "highly recommend" or "waste of time".

**Q3: Which test sentences were classified incorrectly?**
With the original model two sentences were classified incorrectly:
- "i hated the film" was predicted Positive but should be Negative
- "a boring and bad story" was predicted Positive but should be Negative
The model predicted everything as Positive because it had too little training 
data to confidently identify negative patterns.

**Q4: Why might "the movie was not good" be difficult for a bag-of-words model?**
A bag-of-words model only counts individual words and ignores word order. 
It sees the word "good" and associates it with positive sentiment, completely 
missing the word "not" which flips the meaning. Even if "not" is kept in the 
vocabulary, the model has no way of knowing it modifies the word that follows 
it. This is a fundamental limitation of counting words without understanding 
their relationships.

**Q5: What would likely happen if you had 1000 reviews instead of 10?**
With 1000 reviews the model would see many more examples of both positive and 
negative language. It would learn much more reliably that words like "hated", 
"boring" and "terrible" indicate negative sentiment. The overall accuracy would 
improve significantly and the model would be less likely to default to always 
predicting the same class. More data is one of the most effective ways to 
improve any machine learning model.

In [21]:
stop_words = stopwords.words('english')

# with 6 new reviews
corpus_v2 = [
    "the movie was fantastic and i loved every part of it",
    "an absolute masterpiece with brilliant acting",
    "the film was boring and too long",
    "i really enjoyed the story and the visuals",
    "the plot was terrible and the acting was even worse",
    "what a wonderful experience, highly recommend",
    "not worth my time, very disappointing",
    "a truly great film, i will watch it again",
    "the script was weak and the characters were flat",
    "an amazing journey from start to finish",
    "i absolutely loved this film, it was brilliant",
    "terrible movie, i hated every minute of it",
    "stunning visuals and a great story",
    "dull and boring, i fell asleep",
    "one of the best films i have ever seen",
    "awful acting and a weak plot, very bad"
]

categories_v2 = [
    "Positive", "Positive", "Negative", "Positive", "Negative",
    "Positive", "Negative", "Positive", "Negative", "Positive",
    "Positive", "Negative", "Positive", "Negative", "Positive", "Negative"
]


vectorizer_v2 = CountVectorizer(stop_words=stop_words)

X_train_v2 = vectorizer_v2.fit_transform(corpus_v2)
X_test_v2 = vectorizer_v2.transform(test_corpus)

model_v2 = SVC()
model_v2.fit(X_train_v2, categories_v2)

predictions_v2 = model_v2.predict(X_test_v2)

correct_labels = ["Positive", "Negative", "Negative", "Positive"]

print("--- Change 1: Stop words + More data ---")
for sentence, label in zip(test_corpus, predictions_v2):
    print(f"'{sentence}' --> {label}")

correct_count = sum(1 for p, a in zip(predictions_v2, correct_labels) if p == a)
print(f"\nCorrectly classified: {correct_count} out of 4")

--- Change 1: Stop words + More data ---
'the movie was great' --> Positive
'i hated the film' --> Positive
'a boring and bad story' --> Positive
'absolutely loved it' --> Positive

Correctly classified: 2 out of 4


In [22]:
from sklearn.naive_bayes import MultinomialNB

# Reusing corpus_v2 , categories_v2 

vectorizer_v3 = CountVectorizer(stop_words=stop_words)

X_train_v3 = vectorizer_v3.fit_transform(corpus_v2)
X_test_v3 = vectorizer_v3.transform(test_corpus)

model_v3 = MultinomialNB()
model_v3.fit(X_train_v3, categories_v2)

predictions_v3 = model_v3.predict(X_test_v3)

print("--- Change 2: MultinomialNB + Stop words ---")
for sentence, label in zip(test_corpus, predictions_v3):
    print(f"'{sentence}' --> {label}")

correct_count = sum(1 for p, a in zip(predictions_v3, correct_labels) if p == a)
print(f"\nCorrectly classified: {correct_count} out of 4")

--- Change 2: MultinomialNB + Stop words ---
'the movie was great' --> Positive
'i hated the film' --> Negative
'a boring and bad story' --> Negative
'absolutely loved it' --> Positive

Correctly classified: 4 out of 4


In [11]:
print("=" * 45)
print("       MODEL IMPROVEMENT SUMMARY")
print("=" * 45)
print(f"Original  (SVC, no stopwords, 10 reviews): 2/4")
print(f"Change 1  (SVC, stopwords, 16 reviews):    2/4")
print(f"Change 2  (MultinomialNB, stopwords):      4/4")
print("=" * 45)
print("Winner: MultinomialNB with stop words!")

       MODEL IMPROVEMENT SUMMARY
Original  (SVC, no stopwords, 10 reviews): 2/4
Change 1  (SVC, stopwords, 16 reviews):    2/4
Change 2  (MultinomialNB, stopwords):      4/4
Winner: MultinomialNB with stop words!


In [ ]:
import re

def clean_text(text):
    text = text.lower()                        
    text = re.sub(r'[^a-z\s]', '', text)     
    return text

# Clean the data
corpus_clean = [clean_text(sentence) for sentence in corpus_v2]
test_clean = [clean_text(sentence) for sentence in test_corpus]

vectorizer_v4 = CountVectorizer(stop_words=stop_words)

X_train_v4 = vectorizer_v4.fit_transform(corpus_clean)
X_test_v4 = vectorizer_v4.transform(test_clean)

model_v4 = MultinomialNB()
model_v4.fit(X_train_v4, categories_v2)

predictions_v4 = model_v4.predict(X_test_v4)

print("--- Change 3: Clean text + MultinomialNB + Stop words ---")
for sentence, label in zip(test_corpus, predictions_v4):
    print(f"'{sentence}' --> {label}")

correct_count = sum(1 for p, a in zip(predictions_v4, correct_labels) if p == a)
print(f"\nCorrectly classified: {correct_count} out of 4")

--- Change 3: Clean text + MultinomialNB + Stop words ---
'the movie was great' --> Positive
'i hated the film' --> Negative
'a boring and bad story' --> Negative
'absolutely loved it' --> Positive

Correctly classified: 4 out of 4


In [23]:
tricky_sentences = [
    "the movie was not good",       
    "the acting was not bad",       
    "visually impressive but boring", 
    "i wanted to like it"          
]

tricky_clean = [clean_text(s) for s in tricky_sentences]

X_tricky = vectorizer_v4.transform(tricky_clean)
predictions_tricky = model_v4.predict(X_tricky)

human_labels = ["Negative", "Positive", "Negative", "Negative"]

print("--- Part 4: Tricky Sentences ---")
print(f"{'Sentence':<40} {'Predicted':<12} {'Correct':<12} {'correct/incorrect'}")
print("-" * 70)
for sentence, predicted, actual in zip(tricky_sentences, predictions_tricky, human_labels):
    emoji = "correct" if predicted == actual else "incorrect"
    print(f"{sentence:<40} {predicted:<12} {actual:<12} {emoji}")

correct = sum(1 for p, a in zip(predictions_tricky, human_labels) if p == a)
print(f"\nCorrectly classified: {correct} out of 4")

--- Part 4: Tricky Sentences ---
Sentence                                 Predicted    Correct      correct/incorrect
----------------------------------------------------------------------
the movie was not good                   Positive     Negative     incorrect
the acting was not bad                   Negative     Positive     incorrect
visually impressive but boring           Negative     Negative     correct
i wanted to like it                      Positive     Negative     incorrect

Correctly classified: 1 out of 4


## Part 4 – Understanding the Limitations

**Q1: Which sentences were classified incorrectly?**
Three out of four sentences were wrong:
- "the movie was not good" → predicted Positive (should be Negative)
- "the acting was not bad" → predicted Negative (should be Positive)
- "i wanted to like it"    → predicted Positive (should be Negative)

**Q2: Why does the model struggle with negations like "not good" or "not bad"?**
The bag-of-words model only counts individual words — it has no understanding
of word order. So "not good" and "good" both just register as the word "good".
The word "not" was also removed by stop word filtering, making this even worse.
The model cannot understand that "not" completely flips the meaning of the word
that follows it.

**Q3: Why might sarcasm or mixed opinions be difficult?**
Sarcasm uses positive words to express negative meaning, for example
"what a great waste of time". The model sees "great" and predicts Positive,
completely missing the sarcastic tone. Mixed opinions contain both positive
and negative words, so the model just picks whichever side has more words,
losing the nuance of the actual opinion.

**Q4: What information is lost when we only count words?**
- Word order is lost ("not good" vs "good")
- Sentence context is lost (we cannot tell if "like" means enjoyment or comparison)
- Implied meaning is lost ("i wanted to like it" implies disappointment)
- Tone and sarcasm are lost completely

## Part 5 – From Bag-of-Words to Modern NLP

**Q1: What is the main limitation of representing text using only word counts?**
Word counts treat every word independently with no understanding of meaning
or context. The word "good" always looks positive even in "not good". Words
with similar meanings like "fantastic" and "brilliant" are treated as completely
unrelated. The model also has no idea where in the sentence a word appears,
so word order and sentence structure are completely lost.

**Q2: Why might a model that understands word context perform better?**
A context-aware model reads words in relation to their neighbours. For example
it would understand that "not" before "good" flips the sentiment, or that
"wanted to like it" implies disappointment even though "like" sounds positive.
This means it can handle negations, sarcasm, and mixed opinions much more
accurately than simply counting words.

**Q3: What kinds of problems might require more advanced NLP models?**
- Negation: "the movie was not good"
- Sarcasm: "oh great, another boring sequel"
- Mixed sentiment: "visually stunning but emotionally empty"
- Implied meaning: "i wanted to like it"
- Ambiguous words: "the film was sick" (sick = bad or sick = amazing in slang)
These all require understanding context and meaning, not just word counts.

## Final Reflection

The model that worked best in this lab was MultinomialNB combined with 
stop word removal and a slightly larger dataset of 16 reviews. It performed 
significantly better than SVC, achieving 4 out of 4 correct predictions on 
the test sentences. I believe this is because MultinomialNB is specifically 
designed for text classification and works well even with very small datasets 
by calculating word probabilities rather than trying to draw a boundary between 
classes like SVC does.

One of the biggest limitations of this lab was the very small dataset of only 
10 to 16 reviews. With so few examples the model did not have enough data to 
reliably learn the difference between positive and negative language. A real 
sentiment classifier would need thousands of reviews to perform consistently.

The most interesting discovery for me was how badly the model handled negations. 
A simple phrase like "not good" completely fooled the model because it saw the 
word "good" and predicted positive, ignoring the word "not" entirely. This 
showed me that bag-of-words has a fundamental weakness — it cannot understand 
word order or meaning, only word counts.

If I were building a real sentiment classifier I would use a much larger and 
more balanced dataset, apply proper text cleaning, and explore more advanced 
models like BERT that understand the full context of a sentence rather than 
just counting individual words.

In [ ]:
bonus_sentences = [
    "The storyline gripping but in a good way",  # mixed sentiment
    "The movie was terribly good",                # sarcasm/mixed
    "i did not hate the story"                    # negation
]

bonus_clean = [clean_text(s) for s in bonus_sentences]

X_bonus = vectorizer_v4.transform(bonus_clean)
predictions_bonus = model_v4.predict(X_bonus)

print("--- Bonus: Tricky Sentences ---")
for sentence, predicted in zip(bonus_sentences, predictions_bonus):
    print(f"'{sentence}' --> {predicted}")

--- Bonus: Tricky Sentences ---
'The storyline gripping but in a good way' --> Positive
'The movie was terribly good' --> Positive
'i did not hate the story' --> Positive


## Bonus – Tricky Sentences

**1. "The storyline gripping but in a good way" → Predicted: Positive**
This is actually correct! However the model got lucky here. It picked up on 
the word "good" and predicted Positive. It didn't truly understand the mixed 
nature of the sentence — it just happened to land on the right answer for 
the wrong reason.

**2. "The movie was terribly good" → Predicted: Positive**
Interesting result! The model saw both "terribly" (negative) and "good" 
(positive). It predicted Positive, which a human might also consider correct 
since "terribly good" is a way of saying something was extremely good. 
However the model didn't understand the sarcastic/informal phrasing — 
it just counted words and "good" won.

**3. "i did not hate the story" → Predicted: Positive**
This is technically correct — "did not hate" means it was okay/positive. 
But again the model got lucky. It ignored "not" and just saw "hate" as a 
negative word, but then predicted Positive anyway, possibly because "story" 
appeared in positive training examples. The right answer for completely 
the wrong reason!